# 03 · Live play — the REAL test  (the game API)

**Who Wants to Be a PoliMillionaire?** — here the actual game we play, not our dev set.

The `live`-mode counterpart of notebook 01, this is:
- **01 = OFFLINE** ('our own test'): the hand-crafted dev set, gold known, accuracy computed locally.
- **03 = LIVE** ('the real test'): the game server feeds the questions and grades them; `correct`
  only **after** submitting we learn (from `AnswerResult`).

Same pipeline, same `EvalRecord`/JSONL log — only `config.mode='live'` and a logged-in `GameClient` differ.
The single switch is `run_session(pipeline, config, game_client=...)` (D-015).

> ⚠️ **This plays a REAL game.** A leaderboard attempt it consumes and the **30s/question timer it starts**.
> The leaderboard resets ~1 week before the deadline → save serious runs for then. Run the play cell
> deliberately you should — not by a stray Shift+Enter.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 1 · Setup — clone the repo, add `millionaire_client` to the path, install deps
The same clone as notebook 01, plus the course's `millionaire_client` package onto `sys.path` we add
(in the repo at `NLP_assignment_api_client/` it lives).

In [1]:
import os, sys

# The repo, into Colab we clone -- the phase-3 branch we want (Phase 3 lives there, not on main).
REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
BRANCH = '4-rag'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
      # Already cloned (maybe on main) -> fetch, switch to phase-3, pull it.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout {BRANCH} && git pull -q origin {BRANCH}

# Confirm which branch + that default_tools is present (the Phase 3 marker).
!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD)
!grep -q "def default_tools" {REPO_ROOT}/src/tools/__init__.py && echo "✅ default_tools present (phase-3 src)" || echo "❌ still old src"

# Our src tree AND the provided client package, onto the import path both go.
SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
# Into the repo root, change directory we do -- relative paths simpler they become.
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('On sys.path:', SRC, '|', API_CLIENT)

# The provided client, importable it must be -- confirm we do, before any game touch.
from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

Cloning into '/content/NLP'...
remote: Enumerating objects: 313, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 313 (delta 69), reused 68 (delta 32), pack-reused 196 (from 1)
Receiving objects: 100% (313/313), 44.76 MiB | 8.70 MiB/s, done.
Resolving deltas: 100% (113/113), done.
Updating files: 100% (110/110), done.
on branch: 4-rag
✅ default_tools present (phase-3 src)
Repo root: /content/NLP
On sys.path: /content/NLP/src | /content/NLP/NLP_assignment_api_client
millionaire_client, imported it is.


In [2]:
# The inference stack + the client's `requests`, install we do (light it stays).
!pip install -q 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.43.0' sentencepiece einops pyyaml pandas matplotlib requests
print('Installed, the dependencies are.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
Installed, the dependencies are.


In [3]:
import torch

# A GPU, present it must be -- on CPU, a 7B model in 30s answer we cannot.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING -- no GPU. Runtime ▸ Change runtime type ▸ T4 GPU, switch you must.')

CUDA available: True
GPU: Tesla T4


## 2 · Load the LIVE run config
`configs/live.yaml` carries `mode: 'live'`. The competition + game mode, here you choose.

Competitions: 0 Entertainment · 1 History · 2 Science · 3 Maths · 4 Philosophy · 5 News.

In [4]:
from config import RunConfig

# The live config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# Which competition to play + how, here choose it you do.
config.game.competition_id = 0          # 0..5
config.game.game_mode = 'text'          # 'text' | 'speech'
config.run_id = f'live_comp{config.game.competition_id}'

print('mode:', config.mode)
print('competition_id:', config.game.competition_id, '| game_mode:', config.game.game_mode)
print('aim_seconds:', config.game.aim_seconds, '(below the 30s wall, a network margin this keeps)')
print('model:', config.model.name, '|', config.model.quantization)

mode: live
competition_id: 0 | game_mode: text
aim_seconds: 25.0 (below the 30s wall, a network margin this keeps)
model: Qwen/Qwen2.5-7B-Instruct | 4bit


## 3 · Load + warm up the model
Identical to notebook 01 — the cold-start cost paid **before** any timed question, so the 30s wall a cold load never eats.

In [5]:
import time
from inference.engine import TransformersEngine

# Once, the model we load -- the cold-start cost, here we pay it.
t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine 已在显存中,跳过加载。')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded in 224.3s
Warmup in 0.5s


## 4 · Wire the pipeline (same as offline) + log in to the game
DI exactly as notebook 01 (D-006) — only now a logged-in `GameClient` we also build.

Credentials: a PoliMi email as the username; the password from a **Colab secret** named
`poli-millionaire` we read (hardcode it, we must NOT — B3). Add it via the 🔑 panel on the left.

In [8]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from tools import default_tools
from retrieval import WikipediaRetriever

# Phase 4 RAG: a live Wikipedia retriever, built ONLY when config.retrieval.enabled.
# (The ablation -- RAG on vs off -- this one flag toggles. `needs_retrieval` still gates it per question.)
retriever = WikipediaRetriever(top_k=config.retrieval.top_k) if config.retrieval.enabled else None
print('RAG:', f'ON ({config.retrieval.source})' if retriever else 'OFF')

# The collaborators, injected into the pipeline they are (D-006) -- identical to offline.
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=PromptBuilder(strategy=config.prompt_strategy),
    classifier=QuestionClassifier(),
    retriever=retriever,       # Phase 4: RAW Wikipedia extracts; needs_retrieval gates per question, graceful on 429.
    tools=default_tools(),     # Phase 3 ON: the safe-AST calculator. ONLY on maths questions it fires
                               # (needs_calculator gates it) -- on Entertainment etc. a harmless no-op it is.
    latency_budget_s=config.latency_budget_s,
)

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')       # <-- your PoliMi email, fill it you must.
PASSWORD = userdata.get('password')  # <-- a Colab secret, NOT hardcoded (B3).

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions and their ids, list them we do (safe -- no timer this starts).
for c in game_client.list_competitions():
    ml = getattr(c, 'max_levels', '?')
    print('  id=', c.id, '|', c.name, '| max_levels=', ml)


RAG: ON (wikipedia)
Logged in as runjie dai
  id= 0 | Entertainment | max_levels= 15
  id= 1 | Ancient History and Politics | max_levels= 15
  id= 2 | Science and Nature | max_levels= 15
  id= 3 | Maths | max_levels= 15
  id= 4 | Philosophy and Psychology | max_levels= 15
  id= 5 | News | max_levels= 15


## 5 · ▶ Play ONE real game  (consumes a leaderboard attempt + starts the 30s timer)
`run_session` sees `mode='live'` and drives `LiveRunner`: start game → our pipeline answers each
question → submit by integer `Option.id` → log one `EvalRecord` per turn (with `correct` from the server).

⚠️ Deliberately run this. Each turn prints live; the full per-turn record lands in the JSONL log.

In [9]:
from evaluation.runner import run_session

# The ONE switch -- mode='live' it sees, so LiveRunner over the real game it drives.
run_path = run_session(
    pipeline,
    config,
    game_client=game_client,
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
)
print('Live run written to:', run_path)

[1] qid=619 lvl=0 -> A | correct=True | timed_out=False | latency=1.5s (left was 29.902637)
[2] qid=287 lvl=0 -> D | correct=True | timed_out=False | latency=4.1s (left was 29.908394)
[3] qid=286 lvl=0 -> B | correct=False | timed_out=False | latency=1.3s (left was 29.907701)
Live run written to: /content/NLP/experiments/runs/live_comp0


## 6 · Results — how far did we get?
From the same JSONL log as offline it derives. `correct` is None where the server withheld it
(e.g. a timeout). Highest level reached + a per-turn table, here we show.

In [10]:
from evaluation.metrics import load_runs

# The live run, back into a DataFrame we read.
df = load_runs([run_path])

answered = len(df)
known = df[df['correct'].notna()]
n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
acc = (n_correct / len(known)) if len(known) else float('nan')
reached = int(df['level'].dropna().max()) if df['level'].notna().any() else None

print(f'Questions answered: {answered}')
print(f'Correct: {n_correct}/{len(known)}  (accuracy {acc:.0%})')
print(f'Highest level reached: {reached}')

# Per question, the full card -- the choices too, so the wrong ones inspect we can
# (the picked letter, by its option text now we read). Note: `options` only the live
# runs from NOW ON carry; an older log an empty cell shows.
for _, row in df.iterrows():
    opts = row['options'] if 'options' in df.columns and isinstance(row['options'], dict) else {}
    picked = row['predicted_answer']
    mark = {True: 'CORRECT', False: 'WRONG'}.get(row['correct'], 'UNKNOWN')  # None (server withheld) it is.
    print('\n' + '=' * 70)
    print(f"[{mark}] qid={row['qid']} | level={row['level']} | topic={row['topic']} | latency={row['latency_s']:.1f}s")
    print(f"Q: {row['question_text']}")
    if opts:
        for letter, text in opts.items():
            arrow = '   <-- model picked' if letter == picked else ''
            print(f"   {letter}. {text}{arrow}")
    else:
        print(f"   (options not logged for this run)  model picked: {picked}")
    print(f"correct={row['correct']}  |  confidence={row['confidence']}")


Questions answered: 3
Correct: 2/3  (accuracy 67%)
Highest level reached: 0

[CORRECT] qid=619 | level=0 | topic=None | latency=1.5s
Q: Which of the following films was NOT written or directed by Quentin Tarantino?
   A. Inception   <-- model picked
   B. Pulp Fiction
   C. True Romance
   D. Reservoir Dogs
correct=True  |  confidence=1.0

[CORRECT] qid=287 | level=0 | topic=None | latency=4.1s
Q: How does the film The Babadook relate to Jennifer Kent's previous work?
   A. It is unrelated to her previous works
   B. It is a short film she directed
   C. It is a remake of her previous film
   D. It is an extension of her short film Monster   <-- model picked
correct=True  |  confidence=1.0

[WRONG] qid=286 | level=0 | topic=None | latency=1.3s
Q: What term describes the juxtaposition of two seemingly unrelated images in a meme format?
   A. Mosaic
   B. Meme   <-- model picked
   C. Mashup
   D. Montage
correct=False  |  confidence=1.0


## 7 · Observations + next steps

_Fill in after the first real game (feeds the investigation rubric + confirms the API contract):_

- **API contract check (Phase-0 TODO):** did `Option.id` submission + `time_remaining` behave as documented?
- **Latency under the real wall:** any turn near 30s once network RTT is counted? Tune `game.aim_seconds`.
- **Where we fall:** the level + topic of the first wrong answer — a knowledge gap, or a parse slip?
- **Offline vs live gap:** does live accuracy track the ~87% the dev set predicted, or drift?

**Switching modes is a one-liner:** offline ⇄ live is just `config.mode` + whether you pass `game_client`.
For a quick offline re-check: `run_session(pipeline, RunConfig.from_yaml('configs/base.yaml'))` — no game_client.

> Be polite to the server (rate limits are real). Save real leaderboard pushes for after the ~1-week-pre-deadline reset.

## 8 · ▶ Sweep ALL competitions (live) + scores + every wrong question

Plays ONE live game in **each** of the 6 competitions (0–5), back to back, with a polite pause between
(the assignment asks: avoid rapid consecutive requests). Each competition logs its own run dir
(`live_comp{id}`). Then a per-competition **scoreboard** + the consolidated list of **every wrong
question** — text, options, and the letter our model picked — also saved to `experiments/wrong_questions.jsonl`.

> ⚠️ This plays **6 REAL games** (timers + leaderboard). Run it deliberately. The calculator (Phase 3)
> auto-fires on the **Maths** competition; RAG is still off (Phase 4). One game failing (e.g. a rate-limit)
> won't abort the rest — it's logged as a blank row and the sweep continues.

This is the multi-competition counterpart of sections 5–6 (which play a single chosen competition).


In [15]:
from evaluation.runner import run_all_competitions

COMP_NAMES = {0: 'Entertainment', 1: 'Ancient History & Politics', 2: 'Science & Nature',
              3: 'Maths', 4: 'Philosophy & Psychology', 5: 'News'}

# One live game in EVERY competition (0-5), back to back, a polite pause between (PDF: no rapid requests).
# Each competition its own run dir gets (live_comp{id}); one game's failure the rest never sinks.
comp_runs = run_all_competitions(
    pipeline, config, game_client,
    competition_ids=range(6),
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
    pause_s=8.0,
    on_competition=lambda cid: print(f"\n===== ▶ Competition {cid}: {COMP_NAMES.get(cid, '?')} ====="),
)
print('\nSweep done. Run paths:')
for cid, path in comp_runs:
    print(f"  comp {cid} {COMP_NAMES.get(cid, ''):28} -> {path}")



===== ▶ Competition 0: Entertainment =====
[1] qid=549 lvl=0 -> C | correct=True | timed_out=False | latency=1.8s (left was 29.907196)
[2] qid=35 lvl=0 -> B | correct=True | timed_out=False | latency=5.6s (left was 29.906634)
[3] qid=229 lvl=0 -> D | correct=True | timed_out=False | latency=1.5s (left was 29.907169)
[4] qid=392 lvl=0 -> D | correct=True | timed_out=False | latency=1.8s (left was 29.907364)
[5] qid=697 lvl=0 -> C | correct=False | timed_out=False | latency=1.8s (left was 29.906793)

===== ▶ Competition 1: Ancient History & Politics =====
[1] qid=1235 lvl=0 -> A | correct=True | timed_out=False | latency=1.8s (left was 29.906407)
[2] qid=1198 lvl=0 -> B | correct=True | timed_out=False | latency=6.0s (left was 29.906811)
[3] qid=1128 lvl=0 -> D | correct=True | timed_out=False | latency=1.8s (left was 29.90968)
[4] qid=918 lvl=0 -> D | correct=True | timed_out=False | latency=6.1s (left was 29.910219)
[5] qid=991 lvl=0 -> B | correct=True | timed_out=False | latency=1.9

In [16]:
import json
import pandas as pd
from evaluation.metrics import load_runs

rows, wrong_all = [], []
for cid, path in comp_runs:
    if path is None:   # That competition failed (e.g. rate-limited) -- a blank row it gets.
        rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': 0,
                     'correct': 0, 'of': 0, 'accuracy': float('nan'), 'reached_level': None})
        continue
    df = load_runs([path])
    known = df[df['correct'].notna()]
    n_correct = int(known['correct'].astype(float).sum()) if len(known) else 0
    acc = (n_correct / len(known)) if len(known) else float('nan')
    reached = int(df['level'].dropna().max()) if df['level'].notna().any() else None
    rows.append({'comp': cid, 'name': COMP_NAMES.get(cid, ''), 'answered': len(df),
                 'correct': n_correct, 'of': len(known), 'accuracy': acc, 'reached_level': reached})
    for _, r in df[df['correct'] == False].iterrows():   # correct is None (timeout) -> excluded, good.
        wrong_all.append((cid, COMP_NAMES.get(cid, ''), r))

# --- Scoreboard, per competition + overall ---
summary = pd.DataFrame(rows)
print('SCORES BY COMPETITION')
print(summary.to_string(index=False))
tot_c, tot_n = int(summary['correct'].sum()), int(summary['of'].sum())
print((f"\nOVERALL: {tot_c}/{tot_n} = {tot_c / tot_n:.1%}") if tot_n else "\nOVERALL: no graded answers")

# --- Every wrong question: text + options + the letter our model picked ---
print(f"\n{'=' * 72}\nEVERY WRONG QUESTION  ({len(wrong_all)})\n{'=' * 72}")
for cid, name, r in wrong_all:
    opts = r['options'] if 'options' in r and isinstance(r['options'], dict) else {}
    picked = r['predicted_answer']
    print(f"\n[comp {cid} · {name}] qid={r['qid']} level={r['level']}")
    print(f"Q: {r['question_text']}")
    for letter, text in opts.items():
        mark = '   <-- our pick (WRONG)' if letter == picked else ''
        print(f"   {letter}. {text}{mark}")

# --- Save a clean consolidated record of the wrong questions ---
out = os.path.join(REPO_ROOT, 'experiments', 'wrong_questions.jsonl')
with open(out, 'w', encoding='utf-8') as f:
    for cid, name, r in wrong_all:
        f.write(json.dumps({
            'competition_id': cid, 'competition': name, 'qid': r['qid'], 'level': r['level'],
            'question_text': r['question_text'],
            'options': r['options'] if isinstance(r.get('options'), dict) else {},
            'our_wrong_pick': r['predicted_answer'],
        }, ensure_ascii=False) + '\n')
print(f"\nWrong questions saved -> {out}  ({len(wrong_all)} rows)")


SCORES BY COMPETITION
 comp                       name  answered  correct  of  accuracy  reached_level
    0              Entertainment        15        9  15  0.600000              0
    1 Ancient History & Politics        30       26  30  0.866667              0
    2           Science & Nature        31       28  31  0.903226              0
    3                      Maths         9        4   9  0.444444              0
    4    Philosophy & Psychology        65       62  65  0.953846              0
    5                       News         7        2   7  0.285714              0

OVERALL: 131/157 = 83.4%

EVERY WRONG QUESTION  (26)

[comp 0 · Entertainment] qid=286 level=0
Q: What term describes the juxtaposition of two seemingly unrelated images in a meme format?
   A. Mosaic
   B. Meme   <-- our pick (WRONG)
   C. Mashup
   D. Montage

[comp 0 · Entertainment] qid=751 level=0
Q: What is the fundamental principle of M3GAN, the artificial intelligence doll in the film?
   A. It is i

In [18]:
import json, collections
from pathlib import Path

p = Path("experiments/runs/live_comp3/records.jsonl")   # Maths = comp 3
rows = [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]

print("tool_used 分布:", collections.Counter(r.get("tool_used") for r in rows))
print("retrieval_used 分布:", collections.Counter(r.get("retrieval_used") for r in rows))
print("-" * 80)
for r in rows:
    print(f"{r['qid']:>6} tool={str(r.get('tool_used')):<11} " f"correct={str(r.get('correct')):<5} | {r['question_text'][:64]}")

tool_used 分布: Counter({None: 8, 'calculator': 1})
retrieval_used 分布: Counter({False: 5, True: 4})
--------------------------------------------------------------------------------
  6973 tool=None        correct=False | Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the index of <p> in S_5
  6815 tool=calculator  correct=True  | Suppose the graph of $y=f(x)$ includes the points $(1,5),$ $(2,3
  6693 tool=None        correct=False | Which of these statements correctly explains bias?
  6978 tool=None        correct=True  | To survey the opinions of the students at your high school, a re
  6905 tool=None        correct=False | Suppose P is the set of polynomials with coefficients in Z_5 and
  6890 tool=None        correct=True  | Statement 1 | Every nonzero free abelian group has an infinite n
  6800 tool=None        correct=False | Given that a rectangle with length $3x$ inches and width $x + 5$
  7028 tool=None        correct=True  | Statement 1 | If A is connected, the closure of A must be co

In [20]:
me = game_client._client.user.username
for comp in game_client.list_competitions():
    e = game_client._client.leaderboard.find_player(comp.id, me)
    if e:
        print(f"comp {comp.id:>2} {comp.name:<28} reached_level={e.reached_level} score={e.score}")

comp  0 Entertainment                reached_level=15 score=1024000
comp  1 Ancient History and Politics reached_level=12 score=128000
comp  2 Science and Nature           reached_level=13 score=256000
comp  3 Maths                        reached_level=3 score=300
comp  4 Philosophy and Psychology    reached_level=15 score=1024000
comp  5 News                         reached_level=1 score=100
